# Used Car Price Prediction (Regression)

## Overview
This notebook builds an end-to-end machine learning regression pipeline to predict **used car prices** using structured vehicle attributes.

The focus is on:
- understanding raw data
- cleaning columns one by one
- handling missing and inconsistent values correctly
- preparing the dataset for modeling using best practices

## Dataset
- Source: Kaggle – Used Car Price Prediction
- Rows: ~4,000
- Target variable: `price`
- Feature types:
  - Numeric (year, mileage)
  - Categorical (brand, fuel type, transmission, accident history)

Before modeling, the dataset is carefully inspected and cleaned to ensure correctness and reproducibility.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("taeefnajib/used-car-price-prediction-dataset")
print("Path to dataset files:", path)

Using Colab cache for faster access to the 'used-car-price-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/used-car-price-prediction-dataset


In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input/used-car-price-prediction-dataset"):
    for f in files:
        if f.lower().endswith((".csv", ".xlsx", ".parquet")):
            print(os.path.join(root, f))

/kaggle/input/used-car-price-prediction-dataset/used_cars.csv


In [ ]:
import pandas as pd

csv_path = "/kaggle/input/used-car-price-prediction-dataset/used_cars.csv"
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
display(df.head(5))
df.info()

Shape: (4009, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         4009 non-null   object
 1   model         4009 non-null   object
 2   model_year    4009 non-null   int64 
 3   milage        4009 non-null   object
 4   fuel_type     3839 non-null   object
 5   engine        4009 non-null   object
 6   transmission  4009 non-null   object
 7   ext_col       4009 non-null   object
 8   int_col       4009 non-null   object
 9   accident      3896 non-null   object
 10  clean_title   3413 non-null   object
 11  price         4009 non-null   object
dtypes: int64(1), object(11)
memory usage: 376.0+ KB


In [ ]:
summary = []

for col in df.columns:
    s = df[col]
    summary.append({
        "column": col,
        "dtype": str(s.dtype),
        "missing_%": round(s.isna().mean() * 100, 2),
        "unique_values": s.nunique(dropna=True),
        "sample_values": s.dropna().astype(str).head(3).tolist()
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

,column,dtype,missing_%,unique_values,sample_values
0,brand,object,0.00,57,"[Ford, Hyundai, Lexus]"
1,model,object,0.00,1898,"[Utility Police Interceptor Base, Palisade SEL..."
2,model_year,int64,0.00,34,"[2013, 2021, 2022]"
3,milage,object,0.00,2818,"[51,000 mi., 34,742 mi., 22,372 mi.]"
4,fuel_type,object,4.24,7,"[E85 Flex Fuel, Gasoline, Gasoline]"
5,engine,object,0.00,1146,[300.0HP 3.7L V6 Cylinder Engine Flex Fuel Cap...
6,transmission,object,0.00,62,"[6-Speed A/T, 8-Speed Automatic, Automatic]"
7,ext_col,object,0.00,319,"[Black, Moonlight Cloud, Blue]"
8,int_col,object,0.00,156,"[Black, Gray, Black]"
9,accident,object,2.82,2,"[At least 1 accident or damage reported, At le..."


In [ ]:
df_before_price = df.copy()

# Clean price column
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

print(df["price"].dtype)
print(df["price"].describe())

float64
count    4.009000e+03
mean     4.455319e+04
std      7.871064e+04
min      2.000000e+03
25%      1.720000e+04
50%      3.100000e+04
75%      4.999000e+04
max      2.954083e+06
Name: price, dtype: float64


In [ ]:
print(df.columns.tolist())

['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']


In [ ]:
df.columns = df.columns.str.strip()
print("After strip:", df.columns.tolist())

After strip: ['brand', 'model', 'model_year', 'milage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']


In [ ]:
[c for c in df.columns if "mile" in c.lower()]

[]

In [ ]:
df = df.rename(columns={"milage": "mileage"})
print(df.columns.tolist())

['brand', 'model', 'model_year', 'mileage', 'fuel_type', 'engine', 'transmission', 'ext_col', 'int_col', 'accident', 'clean_title', 'price']


In [ ]:
display(df["mileage"].head())

# Clean mileage values
df["mileage"] = (
    df["mileage"]
    .str.replace("mi.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
    .astype(int)
)

print(df["mileage"].dtype)
print(df["mileage"].describe())

,mileage
0,"51,000 mi."
1,"34,742 mi."
2,"22,372 mi."
3,"88,900 mi."
4,"9,835 mi."


int64
count      4009.000000
mean      64717.551010
std       52296.599459
min         100.000000
25%       23044.000000
50%       52775.000000
75%       94100.000000
max      405000.000000
Name: mileage, dtype: float64


In [ ]:
print(df["model_year"].dtype)
print(df["model_year"].describe())

# check suspicious years
print("Min year:", df["model_year"].min())
print("Max year:", df["model_year"].max())

# show any extreme years
display(df[df["model_year"] < 1990][["brand","model","model_year","price"]].head(10))
display(df[df["model_year"] > 2025][["brand","model","model_year","price"]].head(10))

int64
count    4009.000000
mean     2015.515590
std         6.104816
min      1974.000000
25%      2012.000000
50%      2017.000000
75%      2020.000000
max      2024.000000
Name: model_year, dtype: float64
Min year: 1974
Max year: 2024


,brand,model,model_year,price
855,Ford,Bronco,1974,115000.0


,brand,model,model_year,price


In [ ]:
cat_cols = ["fuel_type", "accident", "clean_title"]

for col in cat_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False))


--- fuel_type ---
fuel_type
Gasoline          3309
Hybrid             194
NaN                170
E85 Flex Fuel      139
Diesel             116
–                   45
Plug-In Hybrid      34
not supported        2
Name: count, dtype: int64

--- accident ---
accident
None reported                             2910
At least 1 accident or damage reported     986
NaN                                        113
Name: count, dtype: int64

--- clean_title ---
clean_title
Yes    3413
NaN     596
Name: count, dtype: int64


In [ ]:
# fuel_type cleanup
df["fuel_type"] = df["fuel_type"].replace(["–", "not supported"], pd.NA)
df["fuel_type"] = df["fuel_type"].fillna("Gasoline")

# accident cleanup
df["accident"] = df["accident"].fillna("None reported")

# clean_title cleanup
df["clean_title"] = df["clean_title"].fillna("No")

for col in ["fuel_type", "accident", "clean_title"]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


fuel_type
fuel_type
Gasoline          3526
Hybrid             194
E85 Flex Fuel      139
Diesel             116
Plug-In Hybrid      34
Name: count, dtype: int64

accident
accident
None reported                             3023
At least 1 accident or damage reported     986
Name: count, dtype: int64

clean_title
clean_title
Yes    3413
No      596
Name: count, dtype: int64


In [ ]:
y = df["price"]

# Define features to keep
features = [
    "brand",
    "model_year",
    "mileage",
    "fuel_type",
    "transmission",
    "accident",
    "clean_title"
]

X = df[features]

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

X shape: (4009, 7)
y shape: (4009,)


,brand,model_year,mileage,fuel_type,transmission,accident,clean_title
0,Ford,2013,51000,E85 Flex Fuel,6-Speed A/T,At least 1 accident or damage reported,Yes
1,Hyundai,2021,34742,Gasoline,8-Speed Automatic,At least 1 accident or damage reported,Yes
2,Lexus,2022,22372,Gasoline,Automatic,None reported,No
3,INFINITI,2015,88900,Hybrid,7-Speed A/T,None reported,Yes
4,Audi,2021,9835,Gasoline,8-Speed Automatic,None reported,No


## Model Training & Evaluation

At this stage, the dataset has been cleaned and prepared for modeling.

### Approach
- Split data into training and test sets to prevent data leakage
- Use a preprocessing pipeline:
  - Scale numeric features
  - One-hot encode categorical features
- Train and compare two regression models:
  - **Ridge Regression** (linear baseline)
  - **Random Forest Regressor** (non-linear model)

### Evaluation Metrics
Models are evaluated using:
- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² score
- Training time

This comparison highlights the trade-off between model complexity, performance, and computational cost.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (3207, 7)
Test shape: (802, 7)


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_features = ["model_year", "mileage"]
categorical_features = [
    "brand",
    "fuel_type",
    "transmission",
    "accident",
    "clean_title"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
from sklearn.linear_model import Ridge

numeric_features = ["model_year", "mileage"]
categorical_features = [
    "brand",
    "fuel_type",
    "transmission",
    "accident",
    "clean_title"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

ridge_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=1.0))
    ]
)


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import time

# Train (again, safe)
start_time = time.time()
ridge_model.fit(X_train, y_train)
ridge_train_time = time.time() - start_time

# Predict
y_pred_ridge = ridge_model.predict(X_test)

# Metrics
ridge_mae = mean_absolute_error(y_test, y_pred_ridge)
ridge_mse = mean_squared_error(y_test, y_pred_ridge)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print("Ridge Results")
print("MAE:", ridge_mae)
print("RMSE:", ridge_rmse)
print("R2:", ridge_r2)
print("Train time (s):", ridge_train_time)


Ridge Results
MAE: 24632.397976511587
RMSE: 137112.2464520473
R2: 0.08022687542022078
Train time (s): 0.07693791389465332


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

# Train
start_time = time.time()
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - start_time

# Predict
y_pred_rf = rf_model.predict(X_test)

# Metrics
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_mse = mean_squared_error(y_test, y_pred_rf)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("MAE:", rf_mae)
print("RMSE:", rf_rmse)
print("R2:", rf_r2)
print("Train time (s):", rf_train_time)

Random Forest Results
MAE: 20766.039480465504
RMSE: 133769.9190774321
R2: 0.12452217276549093
Train time (s): 20.408660888671875


In [ ]:
comparison_df = pd.DataFrame({
    "Model": ["Ridge Regression", "Random Forest"],
    "MAE": [ridge_mae, rf_mae],
    "RMSE": [ridge_rmse, rf_rmse],
    "R2": [ridge_r2, rf_r2],
    "Train Time (s)": [ridge_train_time, rf_train_time]
})

display(comparison_df)

,Model,MAE,RMSE,R2,Train Time (s)
0,Ridge Regression,24632.397977,137112.246452,0.080227,0.076938
1,Random Forest,20766.039480,133769.919077,0.124522,20.408661


## Results Summary

Two regression models were trained and evaluated on the cleaned dataset.

- **Ridge Regression** provides a fast and interpretable baseline model.
- **Random Forest Regressor** achieves better predictive performance at the cost of longer training time.

Overall, the results show that non-linear models better capture the structure of used car pricing data, while linear models remain useful for quick baselines and explainability.

Further improvements can be achieved through target transformation, feature engineering, and advanced models.
